# Week 2 — Checkpoint 3: Selection and Execution of Statistical Tests
### Dataset: `cars.csv` (turbo.az — car sale listings, cleaned version)

**Goal:** Execute the hypotheses formulated in Checkpoint 2 using statistical tests appropriate to the data type. For each test, I first explain **why that specific test** was chosen, then execute it. (Formal checking of normality/variance assumptions happens in Checkpoint 5; deep interpretation of p-values/CIs happens in Checkpoint 4.)

This checkpoint covers 3 test types: **t-test** (2 groups), **ANOVA + Bonferroni post-hoc** (5 groups), and, additionally, **chi-square** (association between two categorical variables) — demonstrating full coverage of the main test types.

## 1. Restoring the cleaned dataset

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from itertools import combinations

pd.set_option('display.max_columns', 40)

cols = [
    "id_x", "car_rel_url_x", "datetime_scrape", "price_x", "currency_x", "city",
    "production_year", "engine_displacement_num", "kilometrage_num", "Marka", "Model",
    "Sürətlər qutusu", "Vəziyyəti", "Ötürücü", "Ban növü", "views"
]

df = pd.read_csv("cars.csv", usecols=cols, parse_dates=["datetime_scrape"])
df_dedup = df.sort_values("datetime_scrape").drop_duplicates(subset="car_rel_url_x", keep="last").copy()

exchange_rate = {"AZN": 1.0, "$": 1.70, "€": 1.85}
df_dedup["price_azn"] = df_dedup["price_x"] * df_dedup["currency_x"].map(exchange_rate)

exclude_body_types = ["Yük maşını", "Motosiklet", "Avtobus", "Moped", "Kvadrosikl", "Dartqı", "Mikroavtobus"]
df_clean = df_dedup[~df_dedup["Ban növü"].isin(exclude_body_types)].copy()
df_clean = df_clean[df_clean["price_azn"] >= 1000].copy()

print("Clean dataset shape:", df_clean.shape)


Clean dataset shape: (149478, 17)


## 2. Test 1 — Independent samples t-test (Business Question 1: gearbox type vs price)

**Test choice:** Continuous dependent variable (`price_azn`) + categorical independent variable with 2 levels (`Sürətlər qutusu`) → **independent samples t-test**.

Given the very large sample size (n>37,000 per group), I use **Welch's t-test** as the primary result, since it doesn't assume equal variances (more reliable when variances differ — formal checking happens in Checkpoint 5).

In [2]:
manual = df_clean.loc[df_clean["Sürətlər qutusu"] == "Mexaniki", "price_azn"]
auto = df_clean.loc[df_clean["Sürətlər qutusu"] == "Avtomat", "price_azn"]

print(f"Manual: n={len(manual)}, mean={manual.mean():.0f}, std={manual.std():.0f}")
print(f"Automatic:  n={len(auto)}, mean={auto.mean():.0f}, std={auto.std():.0f}")


Manual: n=37161, mean=10567, std=8188
Automatic:  n=98161, mean=30023, std=34053


In [3]:
t_stat, p_value = stats.ttest_ind(auto, manual, equal_var=False)  # Welch's t-test

# Cohen's d (effect size)
pooled_std = np.sqrt(((len(auto) - 1) * auto.var() + (len(manual) - 1) * manual.var()) / (len(auto) + len(manual) - 2))
cohens_d = (auto.mean() - manual.mean()) / pooled_std

# 95% confidence interval (for the mean difference, using Welch-Satterthwaite dof)
diff = auto.mean() - manual.mean()
se = np.sqrt(auto.var() / len(auto) + manual.var() / len(manual))
dof = (auto.var() / len(auto) + manual.var() / len(manual)) ** 2 / (
    (auto.var() / len(auto)) ** 2 / (len(auto) - 1) + (manual.var() / len(manual)) ** 2 / (len(manual) - 1)
)
t_crit = stats.t.ppf(0.975, dof)
ci_low, ci_high = diff - t_crit * se, diff + t_crit * se

print(f"Welch t-statistic: {t_stat:.2f}")
print(f"p-value: {p_value:.2e}")
print(f"Mean price difference (Automatic - Manual): {diff:.0f} AZN")
print(f"95% Confidence Interval: [{ci_low:.0f}, {ci_high:.0f}] AZN")
print(f"Cohen's d (effect size): {cohens_d:.3f}")


Welch t-statistic: 166.72
p-value: 0.00e+00
Mean price difference (Automatic - Manual): 19456 AZN
95% Confidence Interval: [19227, 19684] AZN
Cohen's d (effect size): 0.664


**Raw result (deep interpretation in Checkpoint 4):** the p-value is essentially 0, and the 95% CI [19227, 19684] AZN doesn't include zero → a statistically significant difference exists. Cohen's d ≈ 0.66 — a medium-to-large effect size (it's important to look at the **magnitude** of the effect, not just the p-value — the subject of Checkpoint 4).

## 3. Test 2 — One-way ANOVA + Bonferroni post-hoc (Business Question 2: brand vs price)

**Test choice:** Continuous dependent variable + categorical independent variable with 5 levels → **one-way ANOVA**.

In [4]:
top5_brands = ["Mercedes", "Hyundai", "Kia", "Toyota", "LADA (VAZ)"]
groups = [df_clean.loc[df_clean["Marka"] == b, "price_azn"].values for b in top5_brands]

for b, g in zip(top5_brands, groups):
    print(f"{b}: n={len(g)}, mean={g.mean():.0f}")


Mercedes: n=25307, mean=25650
Hyundai: n=19658, mean=23864
Kia: n=14619, mean=25765
Toyota: n=14388, mean=30287
LADA (VAZ): n=12859, mean=7264


In [5]:
F_stat, p_value_anova = stats.f_oneway(*groups)

# eta-squared (effect size)
grand_mean = np.concatenate(groups).mean()
ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
ss_total = sum(((g - grand_mean) ** 2).sum() for g in groups)
eta_squared = ss_between / ss_total

print(f"ANOVA F-statistic: {F_stat:.2f}")
print(f"p-value: {p_value_anova:.2e}")
print(f"Eta-squared (effect size): {eta_squared:.3f}")


ANOVA F-statistic: 1942.50
p-value: 0.00e+00
Eta-squared (effect size): 0.082


**ANOVA result:** p ≈ 0 → H₀ (all brands have equal mean price) is rejected. But ANOVA only tells us "at least one difference exists" — to see **which** pairs differ, I move on to Bonferroni-corrected post-hoc tests.

In [6]:
alpha = 0.05
n_comparisons = len(list(combinations(top5_brands, 2)))
alpha_corrected = alpha / n_comparisons
print(f"Number of pairwise comparisons: {n_comparisons}, Bonferroni-corrected α: {alpha_corrected:.4f}")
print()

posthoc_results = []
for b1, b2 in combinations(top5_brands, 2):
    g1 = df_clean.loc[df_clean["Marka"] == b1, "price_azn"]
    g2 = df_clean.loc[df_clean["Marka"] == b2, "price_azn"]
    t_stat, p_raw = stats.ttest_ind(g1, g2, equal_var=False)
    posthoc_results.append({
        "Pair": f"{b1} vs {b2}",
        "Mean diff": round(g1.mean() - g2.mean(), 0),
        "p-value (raw)": p_raw,
        "p < 0.05 (uncorrected)": p_raw < alpha,
        "p < corrected α (Bonferroni)": p_raw < alpha_corrected,
    })

posthoc_df = pd.DataFrame(posthoc_results)
posthoc_df


Number of pairwise comparisons: 10, Bonferroni-corrected α: 0.0050



,Pair,Mean diff,p-value (raw),p < 0.05 (uncorrected),p < corrected α (Bonferroni)
0,Mercedes vs Hyundai,1786.0,4.516971e-13,True,True
1,Mercedes vs Kia,-115.0,6.527356e-01,False,False
2,Mercedes vs Toyota,-4638.0,9.513534e-53,True,True
3,Mercedes vs LADA (VAZ),18386.0,0.000000e+00,True,True
4,Hyundai vs Kia,-1901.0,1.361007e-56,True,True
5,Hyundai vs Toyota,-6424.0,3.022574e-216,True,True
6,Hyundai vs LADA (VAZ),16600.0,0.000000e+00,True,True
7,Kia vs Toyota,-4523.0,3.173328e-99,True,True
8,Kia vs LADA (VAZ),18501.0,0.000000e+00,True,True
9,Toyota vs LADA (VAZ),23024.0,0.000000e+00,True,True


**Result:** 9 out of 10 pairs are significant both uncorrected and after Bonferroni correction. Only the **Mercedes vs Kia** pair shows no significant difference at any level (p=0.65) — meaning these two brands' mean prices are not statistically distinguishable, while every other pair is.

## 4. Additional test — Chi-square (categorical-categorical association)

**Test choice:** This test goes beyond the 2 core business questions but is added to fully demonstrate the methodology. Question: **is there an association between `Sürətlər qutusu` (gearbox type) and `Ötürücü` (drivetrain: front/rear/all-wheel)?** Since both variables are **categorical** (not continuous), t-test/ANOVA don't apply — instead, a **chi-square test of independence** is used.

- **H₀:** `Sürətlər qutusu` and `Ötürücü` are independent of each other (no association).
- **H₁:** the two variables are associated (not independent).

In [7]:
contingency_table = pd.crosstab(df_clean["Sürətlər qutusu"], df_clean["Ötürücü"])
contingency_table


Ötürücü,Arxa,Tam,Ön
Sürətlər qutusu,,,
Avtomat,27499,24826,45836
Mexaniki,13236,3037,20888
Reduktor,359,897,588
Robot,89,180,1258
Variator,112,671,10002


In [8]:
chi2_stat, p_chi2, dof_chi2, expected_freq = stats.chi2_contingency(contingency_table)

# Cramer's V (effect size)
n_total = contingency_table.sum().sum()
cramers_v = np.sqrt(chi2_stat / (n_total * (min(contingency_table.shape) - 1)))

print(f"Chi-square statistic: {chi2_stat:.1f}")
print(f"Degrees of freedom: {dof_chi2}")
print(f"p-value: {p_chi2:.2e}")
print(f"Cramer's V (effect size): {cramers_v:.3f}")
print(f"Minimum expected frequency: {expected_freq.min():.0f} (chi-square validity requires all ≥5)")


Chi-square statistic: 14246.6
Degrees of freedom: 8
p-value: 0.00e+00
Cramer's V (effect size): 0.218
Minimum expected frequency: 302 (chi-square validity requires all ≥5)


**Raw result:** p ≈ 0, and the minimum expected frequency (302) comfortably satisfies chi-square's validity requirement (≥5). Cramer's V ≈ 0.22 — a moderate association. In other words, gearbox type and drivetrain type are statistically significantly, moderately associated (e.g. CVT ("Variator") is mostly found in front-wheel-drive cars, while "Robot" is comparatively rare in rear-wheel-drive cars).

## Checkpoint 3 — Summary

| Test | Variables | Raw result |
|---|---|---|
| Welch's t-test | price_azn (continuous) vs gearbox type (2 levels) | t=166.7, p≈0, 95% CI [19227, 19684] AZN, Cohen's d=0.66 |
| One-way ANOVA | price_azn (continuous) vs brand (5 levels) | F=1942.5, p≈0, η²=0.082 |
| Bonferroni post-hoc | brand pairs (10 comparisons) | 9/10 pairs significant, only Mercedes vs Kia not distinguishable |
| Chi-square | gearbox type vs drivetrain (categorical-categorical) | χ²=14256.5, p≈0, Cramer's V=0.22 |

All three test types (t-test, ANOVA+post-hoc, chi-square) were successfully executed, each selected to match the data type (continuous vs. categorical, 2 groups vs. multiple groups).
